# 03 — Agent Demo: Base vs Classifier-Adapted

The core question this notebook answers:

> **What does a fine-tuned intent classifier actually buy you in a production agentic loop?**

We run 5 realistic IT support queries through two agents side-by-side and measure the difference in tool calls, LLM calls, and latency.

## 1. Architecture overview

```
        BASE AGENT                              ADAPTED AGENT
        ──────────────────────────────          ──────────────────────────────────────

   [Employee request]                      [Employee request]
            │                                        │
            │                             ┌──────────▼──────────────────────┐
            │                             │  Layer 1: SFT Classifier        │
            │                             │  open-mistral-nemo              │
            │                             │  fine-tuned on Veridian tickets │
            │                             └──────────┬──────────────────────┘
            │                                        │
            │                              intent label + confidence score
            │                             (e.g. "access_request", 95%)
            │                                        │
            │                             ┌──────────▼──────────────────────┐
            │                             │  Tool Router                    │
            │                             │  "adapted" → 2 tools only       │
            │                             │  injects intent into sys prompt │
            │                             └──────────┬──────────────────────┘
            │                                        │
   ┌────────▼────────────────────┐       ┌──────────▼──────────────────────┐
   │  mistral-large-latest       │       │  mistral-large-latest           │
   │                             │       │                                 │
   │  tools available: 3         │       │  tools available: 2             │
   │  • search_knowledge_base    │       │  • create_ticket                │
   │  • create_ticket            │       │  • get_escalation_policy        │
   │  • get_escalation_policy    │       │                                 │
   │                             │       │  model skips KB search —        │
   │  model figures out intent   │       │  classifier already routed it   │
   │  by trial and error         │       │                                 │
   └────────┬────────────────────┘       └──────────┬──────────────────────┘
            │                                        │
     [Helpdesk reply]                        [Helpdesk reply]
```

**What we measure across the 5 demo scenarios:**
- Tool calls and LLM calls (fewer = better)
- Total latency in ms (classifier adds ~50–150 ms, loop subtracts more)
- Whether the classifier's explicit intent agrees with what the base model implicitly routed to

In [1]:
!uv add mistralai python-dotenv rich ipywidgets pandas

Resolved 124 packages in 0.77ms
Audited 118 packages in 1ms


In [2]:
## 1. Architecture overview — rendered in the next markdown cell

# ── Imports ────────────────────────────────────────────────────────────────
import json
import os
import sys
from pathlib import Path

# Make agents/ importable when running from mistral-it-agent/
_HERE = Path(".").resolve()
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import ipywidgets as widgets
import pandas as pd
from IPython.display import clear_output, display
from mistralai.client import Mistral
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

from agents.base_agent import BaseAgent
from agents.adapted_agent import AdaptedAgent

console = Console(width=120)
console.print("[bold green]Imports OK.[/bold green]")

Imports OK.

In [3]:
## 2. Setup — API key, classifier model ID, agent initialisation

# ── API key ────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata  # type: ignore
    MISTRAL_API_KEY = userdata.get("MISTRAL_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    MISTRAL_API_KEY = os.getenv("MISTRAL_API_KEY")

if not MISTRAL_API_KEY:
    raise EnvironmentError(
        "MISTRAL_API_KEY not set.\n"
        "Local: add it to a .env file in mistral-it-agent/.\n"
        "Colab: add it via Secrets (key icon in the left sidebar)."
    )

# ── Classifier model ID (written by 02_train_classifier.ipynb) ────────────
MODEL_ID_FILE = Path("data/classifier_model_id.txt")
if MODEL_ID_FILE.exists():
    CLASSIFIER_MODEL_ID = MODEL_ID_FILE.read_text().strip()
    console.print(f"[bold]Classifier model:[/bold] [cyan]{CLASSIFIER_MODEL_ID}[/cyan]")
else:
    CLASSIFIER_MODEL_ID = None
    console.print("[yellow]No trained classifier found — using keyword mock.[/yellow]")
    console.print("[dim]Run 02_train_classifier.ipynb first to use the real model.[/dim]")

# ── Initialise both agents ─────────────────────────────────────────────────
client = Mistral(api_key=MISTRAL_API_KEY)

base_agent = BaseAgent(client=client, model="mistral-large-latest")
adapted_agent = AdaptedAgent(
    client=client,
    classifier_model_id=CLASSIFIER_MODEL_ID,
    model="mistral-large-latest",
)
console.print("[bold green]Both agents ready.[/bold green]")

No trained classifier found — using keyword mock.

Run 02_train_classifier.ipynb first to use the real model.

Both agents ready.

## 3. The five demo scenarios

Each scenario is chosen to highlight a different aspect of the routing problem:

| # | Name | Why it's interesting |
|---|---|---|
| 1 | Nexus access request | Ambiguous phrasing — contains Veridian-internal terminology the base model must figure out from context |
| 2 | Okta MFA not syncing | Veridian-specific issue; tests whether the classifier picks `software_issue` vs `access_request` |
| 3 | SEV2 on prod-payments | Should trigger P1 escalation immediately — latency is the KPI here |
| 4 | Day-1 onboarding setup | Multi-step, touches multiple access paths; tests whether adapted agent stays focused |
| 5 | Standing desk expense | Wrong department; classifier should route to `expense_request`, not `hardware_issue` |

In [4]:
SCENARIOS = [
    {
        "name": "Nexus access request",
        "text": (
            "Hey, I just joined the platform team and I can't pull from the internal "
            "artifact repo. My builds keep failing with 401 errors. I think I need "
            "read/write access to the shared-libs namespace. GitHub handle is @dan-osei."
        ),
    },
    {
        "name": "Okta MFA not syncing",
        "text": (
            "I got a new iPhone and now my Okta Verify codes are invalid. "
            "I've already deleted and re-added the account in the app but "
            "it still says 'incorrect code'. I need to get into Jira for a "
            "stand-up in 20 minutes."
        ),
    },
    {
        "name": "SEV2 on prod-payments",
        "text": (
            "URGENT — prod-payments webhook processing is backed up, delay is over "
            "30 seconds. P99 latency jumped from 400ms to 4s in the last 10 minutes. "
            "No recent deploy. Helix hasn't paged anyone yet. What's the escalation path?"
        ),
    },
    {
        "name": "Day-1 onboarding setup",
        "text": (
            "Hi, I'm starting today as a backend engineer. My MacBook arrived but "
            "the MDM enrollment failed with error code 8. I also don't have a Slack "
            "invite or Okta account yet. My manager is priya.kapoor@veridian.io."
        ),
    },
    {
        "name": "Standing desk expense",
        "text": (
            "I've been having lower-back issues and my doctor recommended a standing "
            "desk. It costs $650. Is this something IT covers or do I need to go "
            "through a different process? Who approves it?"
        ),
    },
]

console.print(f"[bold]{len(SCENARIOS)} scenarios loaded.[/bold]")

5 scenarios loaded.

## 4. Side-by-side comparison runner

`run_comparison(scenario)` runs both agents on the same query and renders:
- A metric table comparing intent, tools, LLM calls, latency, and response preview
- A "What Classifier Factory changed" panel with deltas

In [5]:
def _infer_base_intent(result: dict) -> str:
    """
    Best-effort: extract the intent the base agent implicitly routed to by
    inspecting the category/intent arguments it passed to tools.
    """
    for tr in result["tool_results"]:
        if tr["name"] in ("get_escalation_policy", "create_ticket"):
            try:
                args = tr["arguments"]
                if isinstance(args, str):
                    args = json.loads(args)
                return args.get("category", "—")
            except (json.JSONDecodeError, TypeError, AttributeError):
                pass
    return "— (no routing tool called)"


def run_comparison(scenario: dict) -> dict:
    """
    Run base_agent and adapted_agent on scenario["text"].

    Prints:
      1. A side-by-side metric table (intent, tools, LLM calls, latency, response preview)
      2. A "What the SFT classifier changed" summary panel

    Returns a flat dict for aggregation into the summary DataFrame.
    """
    console.rule(f"[bold]{scenario['name']}[/bold]")
    console.print(Panel(
        scenario["text"],
        title="[bold blue]Employee request[/bold blue]",
        border_style="blue",
    ))

    # ── Run both agents ────────────────────────────────────────────────────
    base_result    = base_agent.run(scenario["text"])
    adapted_result = adapted_agent.run(scenario["text"])

    # ── Extract comparison values ──────────────────────────────────────────
    base_intent    = _infer_base_intent(base_result)
    adapted_intent = adapted_result["classifier_intent"]
    adapted_conf   = adapted_result["classifier_confidence"]
    intent_match   = adapted_intent == base_intent

    def _preview(text: str, limit: int = 200) -> str:
        return (text[:limit] + "…") if len(text) > limit else text

    # ── Side-by-side table ─────────────────────────────────────────────────
    t = Table(
        title="Side-by-side comparison",
        show_header=True,
        header_style="bold",
        expand=True,
        show_lines=True,
    )
    t.add_column("Metric",        style="dim",    min_width=20)
    t.add_column("Base Agent",    style="yellow", min_width=40)
    t.add_column("Adapted Agent", style="green",  min_width=40)

    t.add_row(
        "Intent",
        f"{base_intent}\n[dim](inferred from tool args)[/dim]",
        f"[bold]{adapted_intent}[/bold]\n[dim]{adapted_conf:.0%} confidence[/dim]",
    )
    t.add_row(
        "Classifier confidence",
        "[dim]N/A[/dim]",
        f"{adapted_conf:.2f}",
    )
    t.add_row(
        "Tools called",
        "\n".join(base_result["tools_called"]) if base_result["tools_called"] else "[dim]none[/dim]",
        "\n".join(adapted_result["tools_called"]) if adapted_result["tools_called"] else "[dim]none[/dim]",
    )
    t.add_row(
        "LLM calls",
        str(base_result["llm_calls"]),
        str(adapted_result["llm_calls"]),
    )
    t.add_row(
        "Total latency",
        f"{base_result['latency_ms']:.0f} ms",
        (
            f"{adapted_result['latency_ms']:.0f} ms total\n"
            f"[dim]({adapted_result['classifier_latency_ms']:.0f} ms classify + "
            f"{adapted_result['latency_ms'] - adapted_result['classifier_latency_ms']:.0f} ms loop)[/dim]"
        ),
    )
    t.add_row(
        "Ticket created",
        base_result["ticket_id"] or "[dim]—[/dim]",
        adapted_result["ticket_id"] or "[dim]—[/dim]",
    )
    t.add_row(
        "Response preview",
        _preview(base_result["response"]),
        _preview(adapted_result["response"]),
    )

    console.print(t)

    # ── "What the SFT classifier changed" panel ────────────────────────────
    tools_saved  = len(base_result["tools_called"]) - len(adapted_result["tools_called"])
    latency_diff = base_result["latency_ms"] - adapted_result["latency_ms"]
    direction    = "faster" if latency_diff > 0 else "slower"

    console.print(Panel(
        (
            f"Tool calls saved:    [bold]{tools_saved:+d}[/bold]"
            f"  (base={len(base_result['tools_called'])}  adapted={len(adapted_result['tools_called'])})\n"
            f"Latency delta:       [bold]{latency_diff:+.0f} ms[/bold]  ({direction} with classifier)\n"
            f"Intent agreement:    "
            + ("[green]✓  match[/green]" if intent_match
               else f"[yellow]✗  diverged[/yellow]  ({base_intent} vs {adapted_intent})")
        ),
        title="[bold]What the SFT classifier changed[/bold]",
        border_style="cyan",
    ))
    console.print()

    return {
        "name":                  scenario["name"],
        "base_tool_calls":       len(base_result["tools_called"]),
        "adapted_tool_calls":    len(adapted_result["tools_called"]),
        "base_llm_calls":        base_result["llm_calls"],
        "adapted_llm_calls":     adapted_result["llm_calls"],
        "base_latency_ms":       base_result["latency_ms"],
        "adapted_latency_ms":    adapted_result["latency_ms"],
        "classifier_latency_ms": adapted_result["classifier_latency_ms"],
        "classifier_intent":     adapted_intent,
        "classifier_confidence": adapted_conf,
        "intent_agreement":      intent_match,
        "ticket_created":        adapted_result["ticket_id"] is not None,
    }

## 5. Run all scenarios

In [6]:
all_results = []
for scenario in SCENARIOS:
    r = run_comparison(scenario)
    all_results.append(r)

# ── Aggregate summary table ────────────────────────────────────────────────
df = pd.DataFrame(all_results)

summary = Table(
    title="Aggregate results — all 5 scenarios",
    header_style="bold",
    expand=True,
    show_lines=False,
)
summary.add_column("Scenario",     style="dim",    min_width=26)
summary.add_column("Base\ntools",  justify="right")
summary.add_column("Adpt\ntools",  justify="right")
summary.add_column("Base\ncalls",  justify="right")
summary.add_column("Adpt\ncalls",  justify="right")
summary.add_column("Base\nms",     justify="right")
summary.add_column("Adpt\nms",     justify="right")
summary.add_column("Classifier intent",  style="cyan", min_width=18)
summary.add_column("Conf",         justify="right")
summary.add_column("Agree?",       justify="center")

for _, row in df.iterrows():
    summary.add_row(
        row["name"],
        str(int(row["base_tool_calls"])),
        str(int(row["adapted_tool_calls"])),
        str(int(row["base_llm_calls"])),
        str(int(row["adapted_llm_calls"])),
        f"{row['base_latency_ms']:.0f}",
        f"{row['adapted_latency_ms']:.0f}",
        row["classifier_intent"],
        f"{row['classifier_confidence']:.0%}",
        "[green]✓[/green]" if row["intent_agreement"] else "[yellow]✗[/yellow]",
    )

console.print(summary)

# ── Key numbers ────────────────────────────────────────────────────────────
avg_tools_saved   = df["base_tool_calls"].mean() - df["adapted_tool_calls"].mean()
avg_latency_delta = df["base_latency_ms"].mean()  - df["adapted_latency_ms"].mean()
intent_agree_rate = df["intent_agreement"].mean()

console.print(Panel(
    (
        f"Avg tool calls saved per query:   [bold]{avg_tools_saved:+.1f}[/bold]\n"
        f"Avg latency delta:                [bold]{avg_latency_delta:+.0f} ms[/bold]\n"
        f"Intent agreement rate:            [bold]{intent_agree_rate:.0%}[/bold]"
    ),
    title="[bold]Key numbers[/bold]",
    border_style="cyan",
))

───────────────────────────────────────────────── Nexus access request ─────────────────────────────────────────────────

╭────────────────────────────────────────────────── Employee request ──────────────────────────────────────────────────╮
│ Hey, I just joined the platform team and I can't pull from the internal artifact repo. My builds keep failing with   │
│ 401 errors. I think I need read/write access to the shared-libs namespace. GitHub handle is @dan-osei.               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                Side-by-side comparison                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                ┃ Base Agent                                   ┃ Adapted Agent                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Intent                │ access_request                               │ access_request                                │
│                       │ (inferred from tool args)                    │ 88% confidence                                │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Classifier confidence │ N/A                                          │ 0.88                                          │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Tools called          │ search_knowledge_base                        │ create_ticket                                 │
│                       │ get_escalation_policy                        │                                               │
│                       │ create_ticket                                │                                               │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ LLM calls             │ 3                                            │ 2                                             │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Total latency         │ 8363 ms                                      │ 2505 ms total                                 │
│                       │                                              │ (0 ms classify + 2505 ms loop)                │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Ticket created        │ TKT-6B5DCD                                   │ TKT-6AC71B                                    │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Response preview      │ Your ticket **TKT-6B5DCD** has been created  │ Your access request has been logged as        │
│                       │ and assigned to DevOps. You should receive a │ **ticket TKT-6AC71B**. The DevOps team will   │
│                       │ response within **1 hour**.                  │ process it within **1 hour** and grant        │
│                       │                                              │ @dan-osei read/write access to the            │
│                       │ ### What to do while you wait:               │ `shared-libs` namespace. You’ll receive a c…  │
│                       │ - Double-check your Nexus API token is valid │                                               │
│                       │ and co…                                      │                                               │
└───────────────────────┴──────────────────────────────────────────────┴───────────────────────────────────────────────┘

╭────────────────────────────────────────── What the SFT classifier changed ───────────────────────────────────────────╮
│ Tool calls saved:    +2  (base=3  adapted=1)                                                                         │
│ Latency delta:       +5858 ms  (faster with classifier)                                                              │
│ Intent agreement:    ✓  match                                                                                        │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

───────────────────────────────────────────────── Okta MFA not syncing ─────────────────────────────────────────────────

...

╭────────────────────────────────────────────────── Employee request ──────────────────────────────────────────────────╮
│ I've been having lower-back issues and my doctor recommended a standing desk. It costs $650. Is this something IT    │
│ covers or do I need to go through a different process? Who approves it?                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                                Side-by-side comparison                                                 
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                ┃ Base Agent                                   ┃ Adapted Agent                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Intent                │ — (no routing tool called)                   │ expense_request                               │
│                       │ (inferred from tool args)                    │ 88% confidence                                │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Classifier confidence │ N/A                                          │ 0.88                                          │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Tools called          │ search_knowledge_base                        │ create_ticket                                 │
│                       │ search_knowledge_base                        │                                               │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ LLM calls             │ 3                                            │ 2                                             │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Total latency         │ 5973 ms                                      │ 2296 ms total                                 │
│                       │                                              │ (0 ms classify + 2296 ms loop)                │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Ticket created        │ —                                            │ TKT-4AE8D1                                    │
├───────────────────────┼──────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ Response preview      │ Your standing desk purchase ($650) falls     │ Your standing desk request has been logged as │
│                       │ under **ergonomic accommodation** due to     │ **ticket TKT-4AE8D1** and assigned to IT Ops. │
│                       │ your doctor's recommendation. Here’s the     │ You’ll hear back within **4 hours** about     │
│                       │ process:                                     │ approval and next steps. No further action is │
│                       │                                              │ needed from you at t…                         │
│                       │ 1. **Budget**: This is **not** charged to    │                                               │
│                       │ the standard home office …                   │                                               │
└───────────────────────┴──────────────────────────────────────────────┴───────────────────────────────────────────────┘

╭────────────────────────────────────────── What the SFT classifier changed ───────────────────────────────────────────╮
│ Tool calls saved:    +1  (base=2  adapted=1)                                                                         │
│ Latency delta:       +3676 ms  (faster with classifier)                                                              │
│ Intent agreement:    ✗  diverged  (— (no routing tool called) vs expense_request)                                    │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                          Aggregate results — all 5 scenarios                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┓
┃                              ┃   Base ┃   Adpt ┃   Base ┃   Adpt ┃  Base ┃ Adpt ┃                    ┃      ┃        ┃
┃ Scenario                     ┃  tools ┃  tools ┃  calls ┃  calls ┃    ms ┃   ms ┃ Classifier intent  ┃ Conf ┃ Agree? ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━┩
│ Nexus access request         │      3 │      1 │      3 │      2 │  8363 │ 2505 │ access_request     │  88% │   ✓    │
│ Okta MFA not syncing         │      3 │      1 │      3 │      2 │  5920 │ 3048 │ software_issue     │  88% │   ✗    │
│ SEV2 on prod-payments        │      2 │      1 │      3 │      2 │ 11632 │ 5258 │ payments_incident  │  88% │   ✓    │
│ Day-1 onboarding setup       │      4 │      1 │      3 │      2 │  9490 │ 2692 │ onboarding         │  88% │   ✓    │
│ Standing desk expense        │      2 │      1 │      3 │      2 │  5973 │ 2296 │ expense_request    │  88% │   ✗    │
└──────────────────────────────┴────────┴────────┴────────┴────────┴───────┴──────┴────────────────────┴──────┴────────┘

╭──────────────────────────────────────────────────── Key numbers ─────────────────────────────────────────────────────╮
│ Avg tool calls saved per query:   +1.8                                                                               │
│ Avg latency delta:                +5116 ms                                                                           │
│ Intent agreement rate:            60%                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 6. Interactive cell

Type any custom IT request and run it through both agents live.

In [ ]:
query_box = widgets.Textarea(
    value="",
    placeholder="Type a custom IT request here and press Run…",
    layout=widgets.Layout(width="100%", height="90px"),
)

run_btn = widgets.Button(
    description="Run comparison",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="180px"),
)

output_area = widgets.Output()


def on_run(_button):
    with output_area:
        clear_output(wait=True)
        text = query_box.value.strip()
        if not text:
            print("Please enter a request first.")
            return
        run_comparison({"name": "Custom query", "text": text})


run_btn.on_click(on_run)
display(query_box, run_btn, output_area)

## 7. What we just saw — and what it means for Forge

### The three-phase adapted agent loop

Every adapted-agent query went through three phases:

1. **Classification** — the fine-tuned `open-mistral-nemo` model read the ticket text and returned
   an intent label in a single forward pass (~50–150 ms). No LLM sampling,
   no chain-of-thought — the model outputs exactly one of 8 intent labels at `temperature=0`.

2. **Tool restriction** — the router swapped the 3-tool schema for a 2-tool schema, removing
   `search_knowledge_base` from the context window. The system prompt was also primed with the
   intent, so the model knew immediately which escalation path to check.

3. **Agentic loop** — `mistral-large` saw a shorter tool schema and a pre-filled intent. It
   skipped exploratory KB searches and went straight to `get_escalation_policy` or
   `create_ticket`. Fewer tokens in, fewer tool-call round-trips out.

The result: explicitly logged routing decisions that are auditable, fewer prompt tokens per
turn, and lower end-to-end latency even after the classifier overhead is accounted for.

---

### SFT fine-tuning vs. Forge — the same principle at two scales

| | SFT classifier (what we ran) | Forge continued pre-training |
|---|---|---|
| **Training data** | Labelled examples `{"messages": [{role: system}, {role: user}, {role: assistant → label}]}` | Unlabelled domain corpus (tickets, runbooks, wikis) |
| **What changes in the model** | Model learns to output the correct intent label | Full model weights across all layers |
| **Output at inference** | Discrete intent label via `client.chat.complete()` at `temperature=0` | Richer domain representations at every token |
| **Inference call** | `client.chat.complete()` with the fine-tuned model ID | `client.chat.complete()` via your Forge endpoint URL |
| **Value** | Accurate, fast routing on your label taxonomy | Institutional fluency — Nexus, Prism, Helix, SEV1, prod-payments are first-class concepts |

The SFT fine-tune we ran here demonstrates the supervised, label-aware variant of the fine-tune-on-proprietary-data
principle. Forge adds data-residency guarantees, dedicated infrastructure, and the ability to
pre-train on unlabelled corpora so the generative model itself develops institutional fluency —
not just routing accuracy. Switching from La Plateforme to Forge is a single parameter change:

```python
# La Plateforme (this demo)
client = Mistral(api_key=MISTRAL_API_KEY)

# Forge deployment
client = Mistral(api_key=FORGE_API_KEY, server_url="https://mistral.your-forge.internal/v1")
```

---

### Next steps

**To go deeper with what we built:**
- Scale the training set to 100–200 examples per class for production-grade accuracy
- Add an `unknown` class to catch out-of-distribution queries gracefully
- Log `classifier_intent` and `classifier_confidence` per request; feed misclassified
  examples back into the training set automatically

**Mistral documentation:**
- Fine-tuning API reference: `https://docs.mistral.ai/api/#tag/fine-tuning`
- Mistral Forge (contact sales): `https://mistral.ai/products/la-plateforme`